# Multi-threading

*Ce document est un texte extrait et enrichi des cours de Jacquelin Charbonnel sur la programmation asynchrone et Olivier Goudet pour le calcul GPU ainsi que du tutoriels python accessible [ici](https://www.python-engineer.com/courses/advancedpython/16-threading/).

Le multi-threading consiste à écrire un programme composé de plusieurs fils d’exécution. Chaque **thread** est matérialisé par une suite d’instructions qui lui est propre. L’exécution d’un tel programme donne lieu à plusieurs tâches qui progressent en même temps.

Un thread est léger, car son contexte se réduit à ses instructions, à un compteur ordinal qui pointe sur la prochaine instruction à exécuter et à la pile des appels de fonctions. Il est donc rapide à créer et à détruire. Sa gestion consomme peu de ressources et partage la mémoire avec les autres threads. La commutation de threads est donc rapide, mais en contre partie leur isolation est faible (partage des variables en particulier).

Le module `threading` est l’un des moyens de faire du multi-threading, dans lequel le programmeur crée explicitement les threads. Pour le programmeur, le thread est un concept abstrait: il peut en créer autant qu’il le souhaite dans son programme. Ensuite, au moment de l’exécuter, le programme est confronté au matériel : suivant l’ordinateur cible, un certain nombre de threads seront ou non distribués sur un certain nombre de coeurs (et à partir de 2 cœurs, il y aura parallélisme).

## Création d'un thread

La création d'un thread s'effectue à l'aide la fonction `threading.Thread()`. Les deux arguments principaux sont donnée par:

- `target`: une fonction a exécuté au lancement du thread.
- `args`: les arguments de la fonction cible. Ce présente sous la forme d'un tuple.

Le thread est lancé à la commande `thread.start()`.

La fonction `thread.join()` empêche le programme de poursuivre au delà dans le code tant que tous les threads ne sont pas conclus.


In [5]:
from threading import Thread, current_thread

def worker():
    print('Task number',current_thread().name ,'is done')

if __name__ == "__main__":        
    threads = []
    num_threads = 10

    # create threads and asign a function for each thread    
    for i in range(num_threads):
        thread = Thread(name = i+1, target = worker)
        threads.append(thread)

    # start all threads
    for thread in threads:
        thread.start()

    # wait for all threads to finish
    # block the main thread until these threads are finished    
    for thread in threads:
        thread.join()

Task number 1 is done
,Task number 2 is done
,Task number 3 is done
,Task number 4 is done
,Task number 5 is done
,Task number 6 is done
,Task number 7 is done
,Task number 8 is done
,Task number 9 is done
,Task number 10 is done


**Exercice 1.** Compléter le programme ci-dessus de façon à ce que le traitement réalisé soit l’écriture `n` fois d’un message, après un nombre de secondes aléatoire. La valeur de `n` sera spécifiée à la création de chaque instance. Vous pouvez utiliser les librairies `time` pour mettre en pause l'exécution des threads et `random` pour générer des valeurs aléatoires.

Le multi-threading permet d'artificiellement créer un parallélisme entres différent sous-programme du code principal. Dans l'exemple ci-dessous, deux threads sont simultanément lancé jusqu'à leur complétion.

In [2]:
import threading
import time

def a():
    c = 0    
    while True:
        c += 1
        print('Thread a is running. Number of execution:', c)
        time.sleep(1)
        global stop_thread_a
        if stop_thread_a:
            break

def b():
    c = 0    
    while True:
        c += 1
        print('Thread b is running. Number of execution:', c)
        time.sleep(1)
        global stop_thread_b
        if stop_thread_b:
            break

thread_a = threading.Thread(target=a)
thread_b = threading.Thread(target=b)

stop_thread_a = False
stop_thread_b = False

thread_a.start()
thread_b.start()

time.sleep(3)

stop_thread_a = True
stop_thread_b = True

thread_a.join()
thread_b.join()

Thread a is running. Number of execution: 1
,Thread b is running. Number of execution: 1
,Thread b is running. Number of execution:Thread a is running. Number of execution: 2
, 2
,Thread b is running. Number of execution: 3
,Thread a is running. Number of execution: 3
,Thread b is running. Number of execution: 4
,Thread a is running. Number of execution: 4


**Exercice 2.** Créer un programme qui affiche les entiers consécutifs et intégrer un thread de fond qui stoppe le programme dès lors qu'une touche est entrée. On pourra utiliser la fonction `input`.

## Verrou et partage mémoire

L'exécution de plusieurs threads en parallère implique une compétition entres les différents sous-programmes. Ces derniers partageant le même espace mémoire, il est important de faire attention lors de l'écriture des variables globales. Le programme ci-dessous tente d'incrémenter une variable unique de 1 dont chaque pas est réalisé par un thread différent.

In [9]:
from threading import Thread
import time

# all threads can access this global variable
database_value = 0

def increase():
    global database_value # needed to modify the global value

    # get a local copy (simulate data retrieving)
    local_copy = database_value

    # simulate some modifying operation
    local_copy += 1
    time.sleep(0.1)

    # write the calculated new value into the global variable
    database_value = local_copy


if __name__ == "__main__":

    print('Start value: ', database_value)

    t1 = Thread(target=increase)
    t2 = Thread(target=increase)

    t1.start()
    t2.start()

    t1.join()
    t2.join()

    print('End value:', database_value)

    print('end main')

Start value:  0
,End value: 1
,end main


**Exercice 3.** Qu'observe t'on à la sortie du programme ? Comment expliquez-vous ce résultat ?

Pour éviter ce problème, il est possible d'isoler l'exécution d'une partie du thread afin de s'assurer que cette dernière soit complétée avant que le programme passe à une autre tâche. On encapsule pour ce faire la partie du code concernée dans un protocol de gestion des contextes `lock`.   

In [9]:
from threading import Thread, Lock

def worker(lock):
    #part of the task code that has to be sequentially executed 
    with lock:
        # ...

if __name__ == "__main__": 
    
    lock = Lock()
    thread = Thread(target=worker, args=[lock])
    thread.start()
    thread.join()

<class 'IndentationError'>: expected an indented block after 'with' statement on line 5 (<ipython-input-9-b7062c471d00>, line 8)

**Exercice 4.1**  
1) Utiliser un verrou sur le programme d'incrémentation pour corriger l'erreur d'écriture de la variable `database_value`.

2) Réaliser un programme qui calcule la somme des `n` premiers entiers, chaque addition étant réalisée par un thread. Par exemple, la somme des 100 premiers entiers sera calculée par 100 threads, chacun d’eux ajoutant un entier à la somme. Ajouter une temporisation de quelques milli-secondes à l’entrée de la méthode d’incrémentation.


L'utilisation des verrous peut également se rattacher à une classe et permettre l'exécution mutiple d'actions sur cette dernière, sans risquer un accès partagé aux variables. Dans l'exemple ci-dessous, le verrou est déclarer dans la classe. 

In [ ]:
from threading import Thread, Lock

class account():
    def __init__(self, val = 0):
        self.value = val
        self.account_lock = Lock()

    def action(self):
        with self.account_lock
            #...

**Exercice 4.2**  
A l'aide de l'exemple ci-dessus, créez un exemple de gestion de comptes dans lequel des agents extérieurs peuvent modifier la valeur du compte. Chaque action de l'agent sera effectué par un thread afin de déposer ou retirer de l'argent, dont on simulera l'exécutation par un temps aléatoire.

## File d'attente

L'opérateur `queue` permet d'ordonner et de hiérarchiser une suite de tâches à accomplir qui est pensée pour une implémentation en parallèle. Les différentes méthodes principales sont:

- `q.get()` : retire et renvoie le premier élément. Par défaut, il bloque jusqu'à ce que un élément soit disponible.
- `q.put(item)` : place l'élément à la fin de la file d'attente.
- `q.task_done()` : indique qu'une tâche précédemment mise en file d'attente est terminée. A appeler pour chaque utilisation de `get()` lorsque la tâche est terminée.
- `q.join()` : bloque jusqu'à ce que tous les éléments de la file d'attente aient été obtenus et traités (task_done() a été appelé pour chaque élément).
- `q.empty()` : retourne `True` si la file d'attente est vide.

Le [code](https://www.python-engineer.com/courses/advancedpython/16-threading/) ci-dessous montre un exemple d'utilisation des files d'attentes où 20 tâches sont assignées dans la file et exécuter par des threads différents. A noter que l'argument `thread.daemon` permet de terminer et libérer l'espace mémoire du thread une fois ce dernier terminé (un thread peut toujours tourner en fond si sa tâche n'est pas remplie, même après la fin du programme principal). 


In [17]:
from threading import Thread, Lock, current_thread
from queue import Queue

def worker(q, lock):
    while True:
        value = q.get()  # blocks until the item is available

        # do stuff...
        with lock:
            # prevent printing at the same time with this lock
            print(f"in {current_thread().name} got {value}")
        # ...

        # For each get(), a subsequent call to task_done() tells the queue
        # that the processing on this item is complete.
        # If all tasks are done, q.join() can unblock
        q.task_done()


if __name__ == '__main__':
    q = Queue()
    num_threads = 10
    lock = Lock()

    for i in range(num_threads):
        t = Thread(name=f"Thread{i+1}", target=worker, args=(q, lock))
        t.daemon = True  # empty memory when the main thread is over
        t.start()

    # fill the queue with items
    for x in range(20):
        q.put(x)

    q.join()  # Blocks until all items in the queue have been gotten and processed.

    print('main done')

in Thread2 got 0
,in Thread2 got 8
,in Thread2 got 9
,in Thread2 got 10
,in Thread2 got 11
,in Thread2 got 12
,in Thread2 got 13
,in Thread2 got 14
,in Thread2 got 15
,in Thread10 got 7
,in Thread10 got 19
,in Thread6 got 3
,in Thread7 got 6
,in Thread9 got 5
,in Thread3 got 1
,in Thread2 got 16
,in Thread4 got 2
,in Thread1 got 17
,in Thread5 got 18
,in Thread8 got 4
,main done


**Exercice 5.** Créer un programme qui exécute un nombre `m` de tâches, stockées dans une file d'attente, avec `n` threads pré-déterminés dont les temps d'exécution sont aléatoires. Pour valider la complétion d'une tâche, chaque thread devra également incrémenter une variable globale `database_value` de 1. Afficher la complétion des threads et le numéro de leur tâches dans la file à chaque instant. Qu'elle valeur finale de `database_value` obtenez-vous?

# Multi-processing

Le multi-processing consiste à exécuter, en même temps et indépendamment, un ensemble de tâches, chacune dans son contexte. L’isolation est forte : les tâches s’exécutent dans des espaces mémoire séparés, et ne partagent pas leurs variables. Elles possèdent chacunes leur propre interpéteur Python. Conséquence, les contextes peuvent être volumineux, leur gestion (création, destruction et commutation) peut être coûteuses (overhead).

Le module `multiprocessing` offre le moyen de faire du multi-processing en Python, en offrant au programmeur la possibilité de créer explicitement les tâches (sous forme d’instances de classe). L'exécution de processus en parallèle est très similaire à celle des tâches précédentes. 


In [4]:
from multiprocessing import Process, current_process 
import os 

def worker():
    print('Process',current_process().name ,'is done')

if __name__ == "__main__":        
    processes = []
    num_process = os.cpu_count()

    # create processes and asign a function for each process    
    for i in range(num_process):
        process = Process(name = i+1, target = worker)
        processes.append(process)

    # start all processes
    for process in processes:
        process.start()

    # wait for all processes to finish
    # block the main process until these processes are finished    
    for process in processes:
        process.join()

Process  1Process is done2
,Process  is doneProcess
,3  4Process is doneis done 
,
,5 Process is done
,6 Processis done 
,7 Process is done
,8 is done


**Exercice 6.** On remarque que dans l'exécution du code précédent, l'impression des numéros des processus semble désordonnée. D'après ce que vous avez vu dans le cadre du multi-threading, pourquoi et comment remédier au problème? 

## Communication inter-processus

Les processus ne partagent pas d’espace commun (donc pas de variables partagées), ils doivent échanger leur données en communiquant.

### multiprocessing.Pipe

C’est un tuyau, un canal de communication bidirectionnelle entre 2 processus, leur permettant d’échanger des données et qui évite l'utilisation de locks. La classe `multiprocessing.Pipe()` renvoie une paire d’objets de type Connection représentant les 2 extrémités du tuyau.


In [1]:
from multiprocessing import Pipe

(connect1,connect2) = Pipe() # if the argument duplex is False, then the pipe is uni-directional.

print(type(connect1).__name__)  # ->  'Connection 1'
print(type(connect2).__name__)  # ->  'Connection 2'

Connection
,Connection


Un object `connection` possède (entre autres) les méthodes `send()` qui envoie des données dans le tuyau, `recv()` qui récupère les données du tuyau, `poll()` qui indique si des données sont présentes dans le tuyau, et `close()` qui ferme la connexion.

Chacun des 2 processus embarque l’un des 2 objets `connection`:

In [2]:
from multiprocessing import Pipe

(connect1,connect2) = Pipe()

# send an object
connect1.send('Hello world')

# receive an object
object = connect2.recv()

print(object)

# check if there is data to receive
if connect2.poll():
    print('Data incoming!')
else:
    print('Nothing to declare')

Hello world
,Nothing to declare


**Exercice 7.**  Créer un processus émetteur qui générera des nombres aléatoires et les enverra à un autre processus via une connection. Créer également un processus récepteur qui recevra les nombres envoyés par l'autre processus et les rapportera.

### multiprocessing.queue

La classe `multiprocessing.Queue` implémente l’échange de données entre processus. Elle s’utilise aussi simplement qu’une liste. Ce type est bien adapté au modèle producteur/consommateur. Les processus qui veulent échanger partagent une même instance de `multiprocessing.Queue`.
	
Le pipe est un concept de plus bas niveau que la queue, il nécessite la création explicite de connexions entre 2 processus. Conséquence, le tuyau est plus efficace et plus rapide, mais il est limité à 2 processus. Une queue est plus abstraite, elle est manipulée comme une variable liste partagée entre plusieurs processus. Elle n’est pas limités à 2 processus.

Une file d'attente peut avoir une taille donnée :

In [ ]:
# unlimited sized queue
queue = multiprocessing.Queue()

# 100 sized queue
queue = multiprocessing.Queue(maxsize=100)

On peut tester sa taille `qsize()`, si elle est vide `empty()` ou pleine `full()`. L’ajout de valeurs se fait avec `put()`, et le retrait avec `get()`.

In [ ]:
import multiprocessing as mp

q = mp.Queue()

for i in range(10):
  q.put(i,block=False)
...

while q.qsize()>0:
  item = q.get(block=True)
  print(item)

Par défaut, ces opérations sont bloquantes. Donc `put()` attend qu’il y ait de la place pour déposer la valeur:

In [ ]:
queue.put(item, block=True)
queue.put(item)               # equivalent

et `get()` attend qu’il y ait quelque chose à prendre:

In [ ]:
item = queue.get(block=True)
item = queue.get()            # equivalent

On peut aussi les utiliser dans un mode non bloquant, auxquel cas `get()` et `put()` lancent une exception en cas d’impossibilité. Dans ce mode, `put()` lance une `Full` exception lorsque la file est pleine:

In [ ]:
try:
  queue.put(item, block=False)
except queue.Full:
  # ...

et `get()` lance une `Empty` exception lorsque la file est vide:

In [ ]:
try:
  item = queue.get(block=False)
except queue.Empty:
  # ...

On a aussi un mode mixte, dans lequel l’exception est lancée après un certain temps:

In [ ]:
try:
  queue.put(item, timeout=5)
except queue.Full:
  # ...

In [ ]:
try:
  item = queue.get(timeout=10)
except queue.Empty:
  # ...

On ne doit pas gérer la concurrence par soi-même. Par exemple, ce code:

In [ ]:
if not queue.full():
  queue.put(item, block=False)

n’est pas correct dans un environnement d’exécution concurrent (car il est interruptible). 

**Exercice 8.** Faire un exemple de programme de remplissage et récupération de données dans une file d'attente de taille finie `n`. Chaque action s'effectuera après un temps aléatoire et on souhaite que ces dernières attendent un temps donné avant de renvoyer un erreur. Vous pouvez vous aider des fonctions suivantes qui sont des exemples de programmes producteur/consommateur. 

In [ ]:
def producteur(queue):
    print('Producteur: début', flush=True)
    for i in range(10):
        value = random()
        sleep(value)
        queue.put(value)
    queue.put(None)
    print('Producteur: fin', flush=True)

def consommateur(queue):
    print('Consommateur: début', flush=True)
    while True:
        item = queue.get()
        if not item:
            break
        print(f'>retire {item}', flush=True)
    print('Consommateur: fin', flush=True)

## Calcul matriciel et GPU

Dans le cadre de calcul en très hautes dimensions, le calcul sur GPU (Graphics Processing Unit) peut se réveler particulièrement efficace si on est capable de "vectoriser" l'implémentation. De la même manière qu'un CPU possède plusieurs coeurs qui rendent possible le calcul parallèle (multi-processing), un GPU possède beaucoup plus de coeurs mais avec des puissances de calcul bien plus faible. Le ratio quantité/puissance est alors inversé par rapport un CPU mais ce qui le rend plus efficace dans des tâches qui nécessitent beaucoup de petits calculs: rendu 3D, traitement du signal vidéo, deep learning, IA... Le GPU peut donc servir de "co-processeur" de calcul pour le CPU.

### Pytorch

La librairie `torch` fournie un ensemble de fonctions et modules dédiés au calcul GPU et deep learning avec des réseaux de neurones pour les cartes graphiques Nvidia. 

In [5]:
import torch as th 

x=th.zeros((5,3),device='cpu')

print(x)

tensor([[0., 0., 0.],
,        [0., 0., 0.],
,        [0., 0., 0.],
,        [0., 0., 0.],
,        [0., 0., 0.]])


La mémoire entre CPU/GPU n'est pas partagée et il est nécessaire de spécifier le "device" sur lequel on souhaite que le tenseur soit enregistré. On ne peut en particulier demander des opérations entres des tenseurs qui ne se trouve pas sur le même bloc mémoire. On ne peut pas faire une opération entre un tenseur situé sur le CPU et un tenseur situé sur le GPU (ou bien sur deux GPU différents).

In [ ]:
import torch as th 

# tensor transfert from the cpu to the gpu
x = th.zeros((5,3),device='cpu')
x.to("cuda:0")

# tensor initialization on the gpu
x = th.zeros((5,3),device='cuda:0')

**Exercice 9.** 
1) On pose $A$ une matrice de taille $n\times d$ représentant une $n$-échantillon de données multivariées de dimension $d$. A l'aide d'une opération matricielle, calculer les moyennes empiriques pour chaque marginale. On pourra générer $A$ avec la distribution de son choix.

2) Comparer le temps d'execution avec une approche séquentielle, de type boucle `for` en variant les valeurs de $n$ et $d$.

3) Faites de même mais en comparant le temps d'exécution de l'approche matricielle sur CPU et GPU.

4) Refaites la même expérience avec l'estimation de la matrice de variance-covariance.

### Regression et optimisation

La vectorisation des calculs représente un gain important pour les méthodes d'optimisation. Beaucoup de ces approches font appels à des méthodes types descente de gradient et nécessitent donc l'historique des opération arithmétiques pour mettre à jour les paramètres. L'accès à cette information est facilité dans la librairie pytorch avec l'enregitrement successif des opérations appliquées à un tenseur.

NB: en pratique des méthodes d'optimisations sont déjà proposées dans pytorch qui ne nécessitent pas la manipulation de gradients.

In [6]:
import torch

# requires_grad = True -> tracks all operations on the tensor. 
x = torch.tensor([[1.,2.,3.,4.]], requires_grad=True)
y = x + 2

# y was created as a result of an operation, so it has a grad_fn attribute.
# grad_fn: references a Function that has created the Tensor
print(x) # created by the user -> grad_fn is None
print(y)
print(y.grad_fn)

# Do more operations on y
z = y * y * 3
print(z)
z = z.mean()
print(z)

tensor([[1., 2., 3., 4.]], requires_grad=True)
,tensor([[3., 4., 5., 6.]], grad_fn=<AddBackward0>)
,<AddBackward0 object at 0x7dd754387bb0>
,tensor([[ 27.,  48.,  75., 108.]], grad_fn=<MulBackward0>)
,tensor(64.5000, grad_fn=<MeanBackward0>)


Une fois le calcul terminé, il suffit d'appeler `.backward()` pour que tous les gradients soient calculés automatiquement. Le gradient de ce tenseur sera accumulé dans l'attribut `.grad`. Il s'agit de la dérivée partielle de la fonction $z=f(x)$ par rapport au tenseur. Attention, le calcul du gradient n'est en revanche possible que pour des applications réelles.

In [7]:
z.backward()
print(x.grad) # dz/dx

tensor([[4.5000, 6.0000, 7.5000, 9.0000]])


**Exercice 10.** Calculer le gradient de la fonction $f$ précédente et vérifier que la valeur retournée par le programme pour $x=(1,2,3,4)$ est correcte.

**Exercice 11** On rappelle que dans un modèle de régression linéaire $((Y_i,X_i))_{1\leq i\leq n}\subset\mathbb{R}\times\mathbb{R}^p$, on a la relation $Y_i=X_i\beta+\varepsilon_i$ où $\beta$ est le vecteur d'intérêt et $(\varepsilon_i)_{1\leq i\leq n}$ une suite de variable aléatoire indépendantes, centrées et de variance $\sigma^2$ finie. 

1) Calculer l'estimateur des moindres carrées de $\beta$ (voir cours de régression linéaire et modèle gaussien) en faisant varier la taille de l'échantillon et la dimension $p$.
2) Refaites de même par descente de gradient pour le risque quadratique. On pourra s'inspirer du modèle ci-dessous.

In [ ]:
import torch as th 

class regressor:

	# x and y are tensors
	# l stands for the learning parameter
	
	def __init__(self,x,y):
		self.beta = th.rand(x.size(1), requires_grad=True)
		self.score = 0

	def grad_descent(self,epoch,learning):

		for i in range(epoch):

			self.score = # ...
			self.score.backward()
            
		# This is important so that the gradient does not accumulate during the beta update.
			with th.no_grad():
				self.beta -= learning*self.beta.grad
				print(self.score)

			self.beta.grad.zero_()

In [ ]:
import time

async def say(n,what):
  for i in range(n):
    time.sleep(0.1)
    print(what, end=" ", flush=True)

: 